# Sod problem with production hydrodynamic code

We now run the same problem with the PLUTO code, which is a research code used in astrophysical fluid dynamics.

The physics and algorithms included in the PLUTO code are the same we have studied in this course. PLUTO is a finite-volume, Godunov-type code written in C, for the equations of gas dynamics in 1D, 2D and 3D:

$$\mathbf{U}^{n+1}_i = \mathbf{U}^{n}_i + \frac{\Delta t}{\Delta x}\left(\mathbf{F}^{n+1/2}_{i-1/2} - \mathbf{F}^{n+1/2}_{i+1/2}\right)$$

We will use PLUTO to reconstruct the profile inside each cell, solve a Riemann problem at every interface to get the flux, and update the cell averages. Research codes usually add additional choices, e.g.: several reconstructions, several Riemann solvers, several time-marching schemes, several physics modules (hydrodynamics, magnetohydrodynamics, and their relativistic versions), curvilinear geometries, and a long list of optional physics such as cooling, viscosity and thermal conduction.

Since PLUTO is written in the C language, it is quite fast compared to python. It is also parallelised with the MPI library, but we won't use that in today's class.

**References**:

Paper: A. Mignone et al., *PLUTO: A Numerical Code for Computational Astrophysics*, ApJS (2007) 170:228.

Site: https://plutocode.ph.unito.it

Full user guide: https://plutocode.ph.unito.it/userguide.pdf

## 1. Problem definition in PLUTO:

To run CFD simulations in PLUTO we need to create a directory and edit 3 files:

| File | Read at | Contains |
|---|---|---|
| `init.c` | compile time | the initial condition, as C code, and any user-defined boundary |
| `definitions.h` | compile time | the algorithms: physics module, dimensions, geometry, reconstruction, time stepping, equation of state |
| `pluto.ini` | run time | the grid, the final time, the CFL number, the Riemann solver, the boundaries and the output |

Any changes in `definitions.h` or `init.c` requires a recompilation, while anything in `pluto.ini` can be changed and the same executable re-run.

The executable itself is called `pluto` and is built in the problem directory.

## 2. Numerical settings:

PLUTO provides a set of choices that you should already be familiar with.

For the Sod setup we will use is the one that reproduces it:

| PLUTO setting | Our notebooks |
|---|---|
| `RECONSTRUCTION  FLAT` | piecewise constant, `method = "godunov"` |
| `RECONSTRUCTION  LINEAR` | piecewise linear, `method = "plm"` |
| `RECONSTRUCTION  PARABOLIC` | piecewise parabolic, `method = "ppm"` |
| `LIMITER  MINMOD_LIM / MC_LIM / VANLEER_LIM` | the limiters of nb14 and Lecture 18 |
| `TIME_STEPPING  EULER` | interface states with no time centring, first order in time |
| `TIME_STEPPING  CHARACTERISTIC_TRACING` | the traced half step of Lecture 21, slides 9 and 10 |
| `Solver  two_shock` | the CGF solver of nb18, Part II |
| `EOS  IDEAL` | the gamma-law gas, $p = (\gamma - 1)\rho e$ |
| `CFL` | the Courant number in $\Delta t = C\,\Delta x\,max(abs(u) + c)$ |

## 3. Linux requirements:

PLUTO is compiled from source, so the requirements are a C compiler, `make`, and a Python interpreter for the configuration script.

First check with you have `gcc`, `make` and `python`:

```bash
gcc --version | head -1
make --version | head -1
python3 --version
wget --version | head -1
```

If you don't, you can install them following these:

- On a Debian or Ubuntu system:

```bash
sudo apt update

sudo apt install build-essential gcc make python3 python3-pip wget tar
```

`build-essential` already brings `gcc` and `make`.

- On Fedora the equivalent is:

```bash
sudo dnf install gcc make python3 wget tar
```

If you work on macOS, the same steps apply once the Xcode command line tools are installed with `xcode-select --install`.

## 4. Downloading and setting up a problem:

### 4.1. Download the code:

PLUTO is distributed as a tar archive. We use the copy in the course repository, which is version 4.4.

Download it into your `Documents` folder and unpack it there. The archive contains a single directory, `PLUTO`, of about 16 MB compressed.

```bash
cd "$HOME/Documents"

wget -q --show-progress https://raw.githubusercontent.com/wbandabarragan/computational-fluid-dynamics/main/software/pluto-yt.tar.gz

ls -lh pluto-yt.tar.gz
```

Then untar it:

```bash
cd "$HOME/Documents"

tar -xzf pluto-yt.tar.gz

ls PLUTO
```

### 4.2. Add the PLUTO directory to your environment variables:

PLUTO needs the environment variable `PLUTO_DIR` to point at the directory you just unpacked. Writing it into `~/.bashrc` means every new terminal knows it:

```bash
echo 'export PLUTO_DIR="$HOME/Documents/PLUTO"' >> "$HOME/.bashrc"

export PLUTO_DIR="$HOME/Documents/PLUTO"

echo "PLUTO_DIR = $PLUTO_DIR"

ls "$PLUTO_DIR/setup.py"
```

### 4.3. Create a folder for the simulations:

We keep the code and the runs apart: `PLUTO` holds the source, and each problem gets its own directory somewhere else.

Create the folder that will hold the runs of this course.

```bash
mkdir -p "$HOME/Documents/cfd_simulations"

ls -d "$HOME/Documents/cfd_simulations" 
```

With this you are set!

## 5. Sod problem:

The problem directory is in the course repository, packed as a single file: [software/test_problems/Sod.zip](https://github.com/wbandabarragan/computational-fluid-dynamics/blob/main/software/test_problems/Sod.zip).

```bash
cd "$HOME/Documents/cfd_simulations"

wget -q --show-progress https://raw.githubusercontent.com/wbandabarragan/computational-fluid-dynamics/main/software/test_problems/Sod.zip

unzip -q Sod.zip

ls -R Sod
```

If `unzip` is missing, install it with:

```bash
sudo apt install unzip
```

### 5.1. Explore the folder:

```
Sod/
├── init.c                the initial condition of the shock tube
├── definitions.h         the algorithms
├── pluto.ini             grid, time, solver, boundaries and output
└── other_configs/        the 3 reconstructions, as separate configurations
```

The active setup is the one that should reproduce nb18 with 512 cells on $x \in [0, 1]$, $C = 0.8$, $t_{\rm stop} = 0.25$, outflow boundaries, the `two_shock` solver, and parabolic reconstruction with characteristic tracing.

The other 3 configurations in `other_configs` are the 3 reconstructions we compared in nb18:

| Files | nb18 run | RECONSTRUCTION | TIME_STEPPING |
|---|---|---|---|
| `definitions_01.h`, `pluto_01.ini` | `method = "godunov"` | `FLAT` | `EULER` |
| `definitions_02.h`, `pluto_02.ini` | `method = "plm", limiter = "mc"` | `LINEAR` | `CHARACTERISTIC_TRACING` |
| `definitions_03.h`, `pluto_03.ini` | `method = "ppm"` | `PARABOLIC` | `CHARACTERISTIC_TRACING` |

**Don't do this now:**

To use one of them, copy it over the active pair and recompile:

```bash
cd "$HOME/Documents/cfd_simulations/Sod"

cp other_configs/definitions_01.h definitions.h

cp other_configs/pluto_01.ini pluto.ini
```

### 5.2 Configuring the problem:

PLUTO writes the makefile for each problem directory, and it does so from a text menu.

Let's run the configuration script from inside the problem directory, in a terminal:

```bash
cd "$HOME/Documents/cfd_simulations/Sod"

python3 "$PLUTO_DIR/setup.py"
```

The menu has 5 entries, which you move through with the arrow keys and select with `Enter`:

| Entry | What to do |
|---|---|
| `Setup problem` | shows the choices of `definitions.h`. They are already set for you, so go through them and leave them as they are |
| `Change makefile` | pick the architecture file. On Linux choose **`Linux.gcc.defs`**, which builds a serial executable with `gcc`. On macOS choose `Darwin.gcc.defs` |
| `Auto-update` | leave as it is |
| `Save Setup` | writes `makefile` and `sysconf.out` in the problem directory |
| `Quit` | leaves the menu |

This step is interactive, so run it in a terminal rather than in this notebook. When it finishes, the directory holds a `makefile` that points at your `PLUTO_DIR`:

```bash
cd "$HOME/Documents/cfd_simulations/Sod"

ls makefile sysconf.out && echo && head -12 makefile
```

### 5.3 Compiling:

`make` compiles the PLUTO source together with your `init.c` and `definitions.h`, and leaves an executable called `pluto` in the problem directory.

The first build takes a couple of minutes, because the whole source is compiled; later builds only redo what changed.

Whenever you edit `definitions.h`, or swap in one of the other configurations, run `make clean` first: the compiled objects were built against the old header and must be thrown away.


```bash
cd "$HOME/Documents/cfd_simulations/Sod"

make

ls -ltr pluto
```

### 5.4 Running the simulation:

The executable reads `pluto.ini` at run time, so nothing needs recompiling to change the grid, the final time, the solver or the output.

Run it in serial, from the problem directory:

```bash
cd "$HOME/Documents/cfd_simulations/Sod"

./pluto
```

The run prints its configuration, then one line per group of steps with the step number, the time, the timestep and the number of zones, and finally the total time it took. With 512 cells to $t = 0.25$ it takes a few seconds.

### 5.5 What does PLUTO write?

The last section of `pluto.ini` decides what is written to disk:

```bash
[Static Grid Output]

uservar    0
dbl        0.25  -1   single_file
flt       -1.0   -1   single_file
vtk        0.01  -1   single_file
tab       -1.0   -1   
```

The first number is the interval **in time** between two outputs of that format, the second one an interval in **steps**, and `-1` switches a column off. So `dbl 0.25` gives a binary snapshot at the beginning and at the end, `vtk 0.01` gives one VTK file every hundredth of a time unit, and `tab -1.0` means no ASCII output at all.

These are the files you should find in the problem directory:

| File | Contains |
|---|---|
| `data.0000.vtk` ... `data.0025.vtk` | the solution in VTK format, 26 snapshots, one every $0.01$ |
| `data.0000.dbl`, `data.0001.dbl` | the solution in double precision, at $t = 0$ and $t = 0.25$ |
| `vtk.out`, `dbl.out` | the index of each format: file number, time, timestep, step number and the variables stored |
| `grid.out` | the grid: number of cells and the position of every interface |
| `pluto.0.log` | the log of the run, the same text that was printed on the screen |
| `restart.out` | what PLUTO needs to restart the run from the last output |

Each snapshot holds `rho`, `vx1`, `vx2`, `vx3` and `prs`, i.e. **the primitive variables** we have been studying.

## 6. Reading the output in python:

The VTK files are binary, so we open them with `pyvista`, which reads the grid and the data arrays in one call.

Each file holds the primitive variables as cell data, and the internal energy can be calculated from two of them:

$$e = \frac{p}{\left(\gamma - 1\right)\rho}$$

so we can rebuild the same four panels we plotted in nb18.

In [ ]:
# Import libraries
import numpy as np
import matplotlib.pyplot as plt
import pyvista as pv
import pandas as pd

In [ ]:
# Path to the simulation and the adiabatic index of the run
sim_dir = "/Users/wbandabarragan/Documents/cfd_simulations/Sod/"

gamma = 1.4

# Let's look at what one file contains
mesh = pv.read(sim_dir + "data.0025.vtk")

print(mesh)
print(mesh.array_names)

### 6.1 I/O function:

The function below takes the number of the output file and returns the grid and the four quantities we want to plot.

The data arrays are flat, one value per cell, and the grid comes from the bounds of the mesh.

**Reference:**
https://github.com/wbandabarragan/computational-physics-1/blob/main/unit-3/310-simulation-analysis.ipynb

In [ ]:
def io_pluto(number, sim_dir = "./", gamma = 1.4):
    """
    Read a 1D PLUTO VTK file and return x and the primitive variables.
    """

    mesh = pv.read(sim_dir + "data.%04d.vtk" % number)

    # Number of cells and the cell centres along x
    nx = mesh.dimensions[0] - 1

    x_edges = np.linspace(mesh.bounds[0], mesh.bounds[1], nx + 1)
    x = 0.5*(x_edges[:-1] + x_edges[1:])

    # Data arrays, in code units
    rho = pv.get_array(mesh, "rho", preference = 'cell')
    u = pv.get_array(mesh, "vx1", preference = 'cell')
    p = pv.get_array(mesh, "prs", preference = 'cell')

    # Internal energy per unit mass
    e = p/((gamma - 1.)*rho)

    return x, rho, u, p, e

In [ ]:
# Read the simulation times from vtk.out
df_times = pd.read_csv(sim_dir + "vtk.out", sep = r"\s+", header = None)

times = np.array(df_times.iloc[:,1])

# Read the last snapshot
snapshot = 25

x, rho, u, p, e = io_pluto(snapshot, sim_dir, gamma)

### 6.2 Plotting the solution at `tend`:

In [ ]:
def plot_pluto(x, rho, u, p, e, t):
    """
    Density, velocity, pressure and internal energy of a 1D PLUTO run.
    """

    fig, ax = plt.subplots(4, 1, figsize = (7,12), sharex = True)

    for axis, var, lab in zip(ax, [rho, u, p, e], [r"$\rho$", r"$u$", r"$p$", r"$e$"]):
        axis.plot(x, var, "o", color = "blue", ms = 3., label = "PLUTO")
        axis.set_ylabel(lab)
        axis.grid(alpha = 0.3)

    ax[0].legend()
    ax[3].set_xlabel(r"$x$")
    plt.xlim(0., 1.)
    plt.tight_layout()
    plt.show()

In [ ]:
plot_pluto(x, rho, u, p, e, times[snapshot])